In [ ]:

!pip install -q datasets pillow tqdm

In [ ]:

from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
import os
import random

In [ ]:
BASE_DIR = "classification_data"
IMG_SIZE = (224, 224)
SPLITS = {"train": 0.7, "val": 0.15, "test": 0.15}

# 25 classes 
COCO_CLASSES = {
    "person": 0,
    "bicycle": 1,
    "car": 2,
    "motorcycle": 3,
    "airplane": 4,
    "bus": 5,
    "truck": 7,
    "traffic light": 9,
    "stop sign": 11,
    "bench": 13,
    "bird": 14,
    "cat": 15,
    "dog": 16,
    "horse": 17,
    "cow": 19,
    "elephant": 20,
    "bottle": 39,
    "cup": 41,
    "bowl": 45,
    "chair": 56,
    "couch": 57,
    "potted plant": 58,
    "bed": 59,
    "pizza": 60,
    "cake": 61
}

ID_TO_CLASS = {v: k for k, v in COCO_CLASSES.items()}

for split in SPLITS:
    for cls in COCO_CLASSES:
        os.makedirs(os.path.join(BASE_DIR, split, cls), exist_ok=True)


In [ ]:

print("Loading COCO dataset (streaming)...")
dataset = load_dataset("detection-datasets/coco", split="train", streaming=True)

In [ ]:
from collections import defaultdict
from tqdm import tqdm

MAX_CROPS_PER_CLASS = 500
class_counter = defaultdict(int)
all_crops = []

print("Extracting object crops (limited per class)...")

for sample in tqdm(dataset):
    image = sample["image"].convert("RGB")
    boxes = sample["objects"]["bbox"]
    cats  = sample["objects"]["category"]

    for bbox, cid in zip(boxes, cats):
        if cid in ID_TO_CLASS and class_counter[cid] < MAX_CROPS_PER_CLASS:
            x, y, w, h = bbox
            crop = image.crop((int(x), int(y), int(x + w), int(y + h)))
            crop = crop.resize(IMG_SIZE)

            cls_name = ID_TO_CLASS[cid]
            all_crops.append((crop, cls_name))
            class_counter[cid] += 1

    if all(class_counter[cid] >= MAX_CROPS_PER_CLASS for cid in ID_TO_CLASS):
        break

print("Crops per class:")
for cid in ID_TO_CLASS:
    print(ID_TO_CLASS[cid], class_counter[cid])

print("Total crops collected:", len(all_crops))


In [ ]:

random.shuffle(all_crops)
N = len(all_crops)
train_end = int(N * SPLITS["train"])
val_end = train_end + int(N * SPLITS["val"])


train_data = all_crops[:train_end]
val_data = all_crops[train_end:val_end]
test_data = all_crops[val_end:]

In [ ]:

def save_split(data, split_name):
  counters = {}
  for img, cls in tqdm(data):
    counters.setdefault(cls, 0)
    fname = f"{cls}_{counters[cls]}.jpg"
    path = os.path.join(BASE_DIR, split_name, cls, fname)
    img.save(path)
    counters[cls] += 1


print("Saving training crops...")
save_split(train_data, "train")


print("Saving validation crops...")
save_split(val_data, "val")


print("Saving test crops...")
save_split(test_data, "test")

In [ ]:

print("\nSanity check (images per class in TRAIN):")
for cls in COCO_CLASSES:
  count = len(os.listdir(os.path.join(BASE_DIR, "train", cls)))
  print(cls, count)


print("\nClassification dataset preparation COMPLETE ✅")

In [ ]:
!zip -r class_sample.zip classification_data/train


In [ ]:
from google.colab import files
files.download("class_sample.zip")
